# Baseline — NCN & NCNC

Reproduces the author's numbers for the two core models:
- **NCN** (`cn1`): Neural Common Neighbor — aggregates CN embeddings with sum pooling
- **NCNC** (`incn1cn1`): NCN with Completion — estimates missing common neighbors

All 8 aggregation variants are available below.

In [10]:
import torch
free, total = torch.cuda.mem_get_info()
print(f"Free VRAM: {free/1e9:.2f} GB / Total: {total/1e9:.2f} GB")


Free VRAM: 0.03 GB / Total: 25.29 GB


In [4]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()


In [23]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))    # experiments/

from utils.presets import make_args, CORA, CITESEER, PUBMED, COLLAB, PPA, DDI
from utils.engine  import run_experiment, get_device
import torch


In [24]:
device = get_device()
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


GPU  : NVIDIA RTX 4500 Ada Generation
VRAM : 25.3 GB
PyTorch 2.10.0+cu128  |  CUDA: True
GPU memory: 25.3 GB


## Configuration

In [25]:
DATASET   = CORA   # CORA | CITESEER | PUBMED | COLLAB | PPA | DDI
PREDICTOR = 'cn1'  # NCN  variants : cn1 | attncn1 | transcn1 | grucn1
# NCNC variants : incn1cn1 | attnincn1cn1 | transincn1cn1 | gruincn1cn1
# Note: transincn1cn1 on Cora/Pubmed → add use_amp=True, splitsize=32768

args = make_args(DATASET, predictor=PREDICTOR)
print(args)


Namespace(predictor='cn1', nnlayers=3, beta=1.0, cndeg=-1, trndeg=-1, tstdeg=-1, depth=1, splitsize=-1, cnprob=0.0, learnpt=False, increasealpha=False, use_aa=False, use_ra=False, use_amp=False, grad_clip=0.0, use_gru=False, gru_layers=1, use_diff_feat=False, use_degree_feat=False, weight_decay=0.0, attn_temp=1.0, lrscheduler='none', lr_min=1e-06, lr_patience=10, load=None, loadx=False, loadmod=False, save_gemb=False, savex=False, savemod=False, require_cuda=False, dataset='Cora', model='puregcn', hiddim=256, mplayers=1, xdp=0.7, tdp=0.3, gnndp=0.05, gnnedp=0.0, xdropout=0.7, predp=0.05, preedp=0.4, gnnlr=0.0043, prelr=0.0024, batch_size=1152, testbs=8192, ln=True, lnnn=True, jk=True, use_xlin=True, tailact=True, maskinput=True, res=False, twolayerlin=False, probscale=4.3, proboffset=2.8, pt=0.75, alpha=1.0, use_valedges_as_input=False, epochs=100, runs=1)


## Run

In [26]:
results = run_experiment(args)


GPU  : NVIDIA RTX 4500 Ada Generation
VRAM : 25.3 GB


/home/swagnik/swagnik/energy/venv/lib/python3.11/site-packages/torch_geometric/data/in_memory_dataset.py:131: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([torch_geometric.data.data.DataTensorAttr])` to allowlist this global.
  out = fs.torch_load(path)
/home/swagnik/swagnik/CNT/experiments/ogbdataset.py:30: UserWarning: 'train_test_split_edges' is deprecated, use 'transforms.RandomLinkSplit' instead
  data = train_test_split_edges(data, test_ratio, test_ratio)


2708 tensor(2707)
dataset split 
train edge 3696
valid edge 527
valid edge_neg 1055
test edge 1055
test edge_neg 1055
  run 1/1  ep   1/100  loss=1.9647  Hits@100=(0.3700, 0.4066)  (0.1s)


/home/swagnik/swagnik/CNT/experiments/utils/engine.py:60: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(scaler is not None)):


  run 1/1  ep   2/100  loss=1.4111  Hits@100=(0.5693, 0.5858)  (0.1s)
  run 1/1  ep   3/100  loss=1.3513  Hits@100=(0.5636, 0.5716)  (0.1s)
  run 1/1  ep   4/100  loss=1.3531  Hits@100=(0.5901, 0.6218)  (0.1s)
  run 1/1  ep   5/100  loss=1.3165  Hits@100=(0.6471, 0.6787)  (0.1s)
  run 1/1  ep   6/100  loss=1.3036  Hits@100=(0.6983, 0.6929)  (0.1s)
  run 1/1  ep   7/100  loss=1.2390  Hits@100=(0.7211, 0.7308)  (0.1s)
  run 1/1  ep   8/100  loss=1.1332  Hits@100=(0.8008, 0.7706)  (0.1s)
  run 1/1  ep   9/100  loss=1.0172  Hits@100=(0.8159, 0.7896)  (0.1s)
  run 1/1  ep  10/100  loss=0.9480  Hits@100=(0.8292, 0.8322)  (0.1s)
  run 1/1  ep  11/100  loss=0.9275  Hits@100=(0.8311, 0.8474)  (0.1s)
  run 1/1  ep  12/100  loss=0.9169  Hits@100=(0.8444, 0.8360)  (0.1s)
  run 1/1  ep  13/100  loss=0.8841  Hits@100=(0.8634, 0.8436)  (0.1s)
  run 1/1  ep  14/100  loss=0.8684  Hits@100=(0.8786, 0.8493)  (0.1s)
  run 1/1  ep  15/100  loss=0.8681  Hits@100=(0.8861, 0.8445)  (0.1s)
  run 1/1  ep  16/10

## Results

In [27]:
import pandas as pd
metric = results.get('metric', 'Hits@100')
rows = [[f'run {i+1}', f'{v:.4f}', f'{t:.4f}']
        for i, (v, t) in enumerate(results['per_run'])]
df = pd.DataFrame(rows, columns=['Run', f'Val {metric}', f'Tst {metric}'])
df.loc[len(df)] = ['MEAN ± STD',
                   f"{results['val_mean']:.4f} ±{results['val_std']:.4f}",
                   f"{results['tst_mean']:.4f} ±{results['tst_std']:.4f}"]
print(df.to_string(index=False))


       Run   Val Hits@100   Tst Hits@100
     run 1         0.9260         0.8891
MEAN ± STD 0.9260 ±0.0000 0.8891 ±0.0000
